# 03 — Multi-seed evaluation (no deltas)

**Manuscript-primary** feature set. Protocol: `evaluation_protocol.md`.

Loads `data/modelling_landmark_trees.csv`. Shared code is `eval_core.py` (legacy grids + `apply_dense_imputation`). Do not `%run` `02_imputation.ipynb`.

Selection: inner-CV **macro F1**. Headline: 5-seed test 3-class macro F1, two-stage vs direct. Optional later: `--seeds 0-19`.

First pass = five paper families + Dummy. Exploratory extras are a later cell.

Long run. Safe to re-execute: finished seeds are skipped (`resume=True`). Equivalent CLI:

```bash
python revision_work/eval_core.py --which no_deltas --seeds 0-4
```

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'revision_work':
    REV = ROOT
    ROOT = ROOT.parent
else:
    REV = ROOT / 'revision_work'
if str(REV) not in sys.path:
    sys.path.insert(0, str(REV))

from eval_core import (
    CLINICAL_COLS, REVISION_FLAGS, SEEDS, require_boosting, run_evaluation,
    tree_feature_columns,
)

require_boosting()
CSV = REV / 'data' / 'modelling_landmark_trees.csv'
OUT = REV / 'results' / 'no_deltas'
df = pd.read_csv(CSV)
print(df.shape)
print(df['falls_class'].value_counts().sort_index().to_dict())
assert df['PATNO'].is_unique
assert df['falls_class'].notna().all()
for c in tree_feature_columns(use_deltas=False):
    assert c in df.columns, c
for c in CLINICAL_COLS + REVISION_FLAGS:
    assert c in df.columns
print('predictors', len(tree_feature_columns(False)))
print('out', OUT)

## 5-seed run

Writes `results/no_deltas/seed_*.json`, `seeds.csv`, `summary.json`.

In [ ]:
rows, summary = run_evaluation(
    CSV, OUT, use_deltas=False, seeds=SEEDS, resume=True,
)
print(json.dumps(summary, indent=2))

## Headline table

In [ ]:
seeds = pd.read_csv(OUT / 'seeds.csv')
display(seeds)

def _fmt(d):
    return f"{d['mean']:.3f} (SD {d['sd']:.3f}; 95% CI {d['ci_low']:.3f}–{d['ci_high']:.3f})"

print('Two-stage test macro F1 ', _fmt(summary['two_stage_f1_macro']))
print('Direct    test macro F1 ', _fmt(summary['direct_f1_macro']))
print('Dummy     test macro F1 ', _fmt(summary['dummy_f1_macro']))
w = summary['two_stage_minus_direct']
print(f"Two-stage − direct: mean {w['mean_diff']:.3f}, Wilcoxon p={w['p_value']:.4f}")
print('Stage 1 winners', summary['s1_winners'])
print('Stage 2 winners', summary['s2_winners'])
print('Direct winners ', summary['direct_winners'])